In [2]:
import pandas as pd
import numpy as np
import os


In [4]:
TRANS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/transactions_2016_2017.csv"
TRAIN_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_train.csv"
TEST_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_test.csv"
OUTPUT_DIR = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data"
TRAIN_OUT_PATH = f"{OUTPUT_DIR}/train_features.csv"
TEST_OUT_PATH = f"{OUTPUT_DIR}/test_features.csv"
ALL_OUT_PATH = f"{OUTPUT_DIR}/all_features.csv"


In [5]:
print("Loading data...")
required_paths = [TRANS_PATH, TRAIN_PATH, TEST_PATH]
missing_required = [p for p in required_paths if not os.path.exists(p)]
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}")

transactions = pd.read_csv(TRANS_PATH, parse_dates=['order_date', 'pack_date'])
train_target = pd.read_csv(TRAIN_PATH)
test_target = pd.read_csv(TEST_PATH)
print("Data loaded successfully.")


Loading data...


/tmp/ipykernel_60710/3938347944.py:7: DtypeWarning: Columns (0: prod_size) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv(TRANS_PATH, parse_dates=['order_date', 'pack_date'])


Data loaded successfully.


In [ ]:
#The features for each transaction are as follows:
#
#cust_id: unique identifier of the customer placing the order
#order_date: date when the order was placed
#pack_date: date when the order was packed/shipped
#sale_id: unique identifier of the sales transaction (multiple articles can be present in the same transaction)
#sale_discount_applied: monetary value of discount applied to the sale
#sale_revenue: final revenue amount received for this line item after discount
#returned_to_shop_id: identifier of the shop/location where the item was returned (empty if not returned)
#prod_id: unique identifier of the purchased product
#prod_size: shoe size of the product
#prod_web_only: binary flag (1/0) indicating whether the product is sold online only
#prod_season: season or collection code (e.g., W14 = Winter 2014)
#prod_brand: brand name of the product
#prod_title: full commercial product name/title
#prod_color: primary color of the product
#prod_type_1: primary target group or category (e.g., men, women, boys)
#prod_type_2: not included (= "shoes")
#prod_type_3: secondary product category (e.g., sneakers, high shoes)
#prod_type_4: tertiary style classification (e.g., high-top sneakers)
#prod_type_5: additional style descriptor (e.g., boots with velcro, dress boots)
#prod_heel: heel type or heel specification (if known)
#prod_material: main outer material of the shoe (e.g., leather, suede) (if known)
#prod_insole: indicator of specific insole feature (if known)
#prod_print: type of print or pattern (if known)
#prod_comfort_sole: indicator of special comfort sole feature (if known)
#prod_comfort_wear: indicator of enhanced comfort wear feature (if known)
#prod_clasp: type of closing mechanism (e.g., velcro, zipper, lace-up) (if known)
#prod_outlet: indicator how often this product was sold through an outlet channel, higher values indicate that the product appeared more often
#
### NOTES
#REAL_REVENUE = sale_revenue + sale_discount_applied
#GROUPING BY CUST_ID <- MONTHS WHERE THE CUSTOMER MADE PURCHASES
#GROUPING BY CUST_ID <- LENGTH OF RETURNED ITEMS (returned_to_shop_id)
#PROD SIZE <- CATEGORICAL VARIABLE
#PROD_TYPE_1 <- TARGET CATEGORY
#PROD_TYPE_2,3,4,5 <- SPECIFIC PRODUCT CATEGORY (PREFERENCES?)
#
#transactions.groupby('cust_id')['sale_revenue'].sum().sort_values(ascending=False)
#transactions.groupby('cust_id').size().sort_values(ascending=False)
#transactions['prod_type_1'].unique()
##Probably exist some correlation between prod_size and prod_type_1
#transactions.loc[transactions['prod_type_1'] == 'boys', 'prod_size'].value_counts()
#transactions['prod_type_3'].unique() #multinomial
#transactions['prod_type_4'].unique() #multi-label
#transactions['prod_type_5'].unique() #Multinomial
#transactions.columns
#transactions['prod_material'].unique() #multi-label
#transactions['prod_insole'].unique() #Binary variable, but many missing values
#transactions['prod_print'].unique() #multi-label
#transactions['prod_comfort_sole'].unique() #multi-label
#transactions['prod_comfort_wear'].unique() #multi-label
#transactions['prod_clasp'].unique() #multi-label
#transactions['prod_outlet'].unique() #Binary variable
#transactions['prod_heel'].unique() #ordenal categorical variable
#transactions['prod_size'].unique()
#print('Hey')


In [9]:
def add_window_features(trx: pd.DataFrame, customer_features: pd.DataFrame, reference_date: pd.Timestamp):
    windows = [30, 90, 180, 365]

    # base table with one row per customer
    if "cust_id" not in customer_features.columns:
        return customer_features

    for w in windows:
        cutoff = reference_date - pd.Timedelta(days=w)
        w_trx = trx[trx["order_date"] >= cutoff].copy()

        if len(w_trx) == 0:
            tmp = pd.DataFrame({"cust_id": customer_features["cust_id"]})
        else:
            tmp = w_trx.groupby("cust_id").agg(
                **{
                    f"revenue_{w}d": ("sale_revenue", "sum"),
                    f"orders_{w}d": ("sale_id", "nunique") if "sale_id" in w_trx.columns else ("sale_revenue", "size"),
                    f"lines_{w}d": ("sale_revenue", "size"),
                    f"avg_ticket_{w}d": ("sale_revenue", "mean"),
                }
            ).reset_index()

        customer_features = customer_features.merge(tmp, on="cust_id", how="left")

    # trend features
    last_6m = trx[trx["order_date"] >= (reference_date - pd.Timedelta(days=180))]
    prev_6m = trx[(trx["order_date"] < (reference_date - pd.Timedelta(days=180))) &
                  (trx["order_date"] >= (reference_date - pd.Timedelta(days=360)))]

    last_3m = trx[trx["order_date"] >= (reference_date - pd.Timedelta(days=90))]
    prev_3m = trx[(trx["order_date"] < (reference_date - pd.Timedelta(days=90))) &
                  (trx["order_date"] >= (reference_date - pd.Timedelta(days=180)))]

    rev_last_6m = last_6m.groupby("cust_id")["sale_revenue"].sum().rename("revenue_last_6m")
    rev_prev_6m = prev_6m.groupby("cust_id")["sale_revenue"].sum().rename("revenue_prev_6m")
    ord_last_3m = (last_3m.groupby("cust_id")["sale_id"].nunique().rename("orders_last_3m")
                   if "sale_id" in trx.columns else last_3m.groupby("cust_id").size().rename("orders_last_3m"))
    ord_prev_3m = (prev_3m.groupby("cust_id")["sale_id"].nunique().rename("orders_prev_3m")
                   if "sale_id" in trx.columns else prev_3m.groupby("cust_id").size().rename("orders_prev_3m"))

    trend = pd.concat([rev_last_6m, rev_prev_6m, ord_last_3m, ord_prev_3m], axis=1).reset_index()
    customer_features = customer_features.merge(trend, on="cust_id", how="left")

    eps = 1e-6
    customer_features["revenue_trend_ratio_6m"] = (
        customer_features["revenue_last_6m"] / (customer_features["revenue_prev_6m"] + eps)
    )
    customer_features["orders_trend_diff_3m"] = (
        customer_features["orders_last_3m"] - customer_features["orders_prev_3m"]
    )

    return customer_features

def add_cadence_features(trx: pd.DataFrame, customer_features: pd.DataFrame):
    if "sale_id" in trx.columns:
        ord_dates = trx.groupby(["cust_id", "sale_id"], as_index=False)["order_date"].min()
    else:
        ord_dates = trx[["cust_id", "order_date"]].copy()

    ord_dates = ord_dates.sort_values(["cust_id", "order_date"])
    ord_dates["prev_order_date"] = ord_dates.groupby("cust_id")["order_date"].shift(1)
    ord_dates["interpurchase_days"] = (ord_dates["order_date"] - ord_dates["prev_order_date"]).dt.days

    cadence = ord_dates.groupby("cust_id")["interpurchase_days"].agg(
        interpurchase_mean="mean",
        interpurchase_std="std",
        interpurchase_max="max",
    ).reset_index()

    customer_features = customer_features.merge(cadence, on="cust_id", how="left")
    return customer_features

def add_diversity_features(trx: pd.DataFrame, customer_features: pd.DataFrame):
    grp = trx.groupby("cust_id")

    diversity = grp.agg(
        n_unique_brands=("prod_brand", "nunique") if "prod_brand" in trx.columns else ("sale_revenue", "size"),
        n_unique_type1=("prod_type_1", "nunique") if "prod_type_1" in trx.columns else ("sale_revenue", "size"),
        n_unique_type3=("prod_type_3", "nunique") if "prod_type_3" in trx.columns else ("sale_revenue", "size"),
    ).reset_index()

    customer_features = customer_features.merge(diversity, on="cust_id", how="left")

    # brand concentration: share of top brand
    if "prod_brand" in trx.columns:
        brand_counts = trx.groupby(["cust_id", "prod_brand"]).size().rename("cnt").reset_index()
        total_counts = brand_counts.groupby("cust_id")["cnt"].sum().rename("total_cnt")
        top_counts = brand_counts.groupby("cust_id")["cnt"].max().rename("top_brand_cnt")
        conc = pd.concat([total_counts, top_counts], axis=1).reset_index()
        conc["brand_top1_share"] = conc["top_brand_cnt"] / conc["total_cnt"].replace(0, np.nan)
        customer_features = customer_features.merge(conc[["cust_id", "brand_top1_share"]], on="cust_id", how="left")

    return customer_features

def build_features(data, transactions): 
    trx = transactions.copy()

    #Remove duplicates
    before = len(trx)
    trx.drop_duplicates(inplace=True)
    print(f"drop_duplicates   : {before - len(trx):,} removed ({len(trx):,} remain)")

    # Date formats
    trx['order_date'] = pd.to_datetime(trx['order_date'], format='%Y-%m-%d')
    trx['pack_date'] = pd.to_datetime(trx['pack_date'], format='%Y-%m-%d')
    trx['delivery_time'] = (trx['pack_date'] - trx['order_date']).dt.days

    # Curstomer aggregations
    customer_features = trx.groupby('cust_id').agg(
    customer_revenue=('sale_revenue','sum'),
    customer_purchases_number=('sale_id','nunique'),
    n_lines=('sale_revenue', 'size'),
    mean_revenue=('sale_revenue', 'mean'),
    n_products=('prod_id', 'nunique') if 'prod_id' in trx.columns else ('sale_revenue', 'size'),
    mean_discount=('sale_discount_applied', 'mean'),
    ).reset_index()

    # Average delivery time per customer
    delivery = trx.groupby('cust_id')['delivery_time'].mean().reset_index()
    customer_features = customer_features.merge(delivery, on='cust_id', how='left')
    # Extract month of purchase
    month_dummies = pd.get_dummies(trx['order_date'].dt.month, prefix='month_')
    month_dummies['cust_id'] = trx['cust_id']
    month_dummies = month_dummies.groupby('cust_id').sum().reset_index()

    # Total discount per customer
    discount = trx.groupby('cust_id')['sale_discount_applied'].sum().reset_index(name='total_discount')
    customer_features = customer_features.merge(discount, on='cust_id', how='left')
    web_only = trx.groupby('cust_id')['prod_web_only'].sum().reset_index(name='web_only_purchases')
    customer_features = customer_features.merge(web_only, on='cust_id', how='left')

    # Fix product size
    trx['prod_size'] = trx['prod_size'].str.strip()  # remove leading/trailing spaces
    trx['prod_size'] = trx['prod_size'].str.upper()  # convert to uppercase for consistency
    size_map = {'XS': 17, 'S':  36, 'M':  38, 'L':  40,'XL': 42} #map sizes to numeric values
    trx['prod_size'] = trx['prod_size'].replace(size_map) 
    trx['prod_size'] = pd.to_numeric(trx['prod_size'], errors='coerce')

    bins = [0, 30, 36, 44, 60]
    labels = ["kids", "youth", "adult", "large"]
    trx['size_group'] = pd.cut(trx['prod_size'], bins=bins, labels=labels)
    size_dummies = trx['size_group'].astype(str).str.get_dummies().add_prefix('size_')
    size_dummies['cust_id'] = trx['cust_id']
    # How much of each size group each customer bought
    size_dummies = size_dummies.groupby('cust_id').sum().reset_index()

    # Alpha size grouping (keeps explicit XS/S/M/L/XL signal)
    size_alpha = trx['prod_size'].astype(str).str.strip().str.upper()
    alpha_levels = ['XS', 'S', 'M', 'L', 'XL']
    size_alpha = np.where(pd.Series(size_alpha).isin(alpha_levels), size_alpha, 'OTHER')
    size_alpha_dummies = pd.get_dummies(pd.Series(size_alpha, index=trx.index), prefix='size_alpha')
    size_alpha_dummies['cust_id'] = trx['cust_id']
    size_alpha_dummies = size_alpha_dummies.groupby('cust_id').sum().reset_index()

    # Season grouping
    def map_season(x):
        if pd.isna(x):
            return 'unknown'
        if x.startswith('W'):
            return 'winter'
        elif x.startswith('S'):
            return 'summer'
        elif x.startswith('Z'):
            return 'mid_winter'
        elif x.startswith('M'):
            return 'mid_season'
        elif x in ['NOS', 'CONS']:
            return 'basic'
        else:
            return 'other'

    trx['prod_season_grouped'] = trx['prod_season'].apply(map_season)
    season_dummies = pd.get_dummies(trx['prod_season_grouped'], prefix='season')
    season_dummies['cust_id'] = trx['cust_id']
    season_dummies = season_dummies.groupby('cust_id').sum().reset_index()
    customer_features = customer_features.merge(season_dummies, on='cust_id', how='left')

    # Recency
    last_purchase = trx.groupby('cust_id')['order_date'].max()
    reference = trx['order_date'].max()
    recency = (reference - last_purchase).dt.days.reset_index(name='recency_days')


    first_purchase = trx.groupby('cust_id')['order_date'].min()
    customer_lifespan_days = (last_purchase - first_purchase).dt.days.reset_index(name='customer_lifespan_days')
    tenure_days = (reference - first_purchase).dt.days.reset_index(name='tenure_days')

    month_activity = (
        trx.assign(order_month=trx['order_date'].dt.to_period('M'))
        .groupby('cust_id')['order_month'].nunique()
        .reset_index(name='active_months')
    )
    # Merge all features
    customer_features = customer_features.merge(recency, on='cust_id', how='left')
    customer_features = customer_features.merge(customer_lifespan_days, on='cust_id', how='left')
    customer_features = customer_features.merge(tenure_days, on='cust_id', how='left')
    customer_features = customer_features.merge(month_activity, on='cust_id', how='left')
    customer_features = customer_features.merge(size_dummies, on='cust_id', how='left')
    customer_features = customer_features.merge(size_alpha_dummies, on='cust_id', how='left')
    customer_features = customer_features.merge(month_dummies, on='cust_id', how='left')
    
    # Multi-label features
    #,'prod_type_4','prod_type_5''prod_comfort_sole','prod_comfort_wear'
    multiple_labels = ['prod_type_3','prod_material','prod_print','prod_clasp']
    multi_features = []
    # Create multi-label features
    for col in multiple_labels:
        cleaned = (
            trx[col]
            .fillna('unknown')
            .astype(str)
            .str.split(',')
            .apply(lambda vals: ','.join(v.strip() for v in vals))
        )
        dummies = cleaned.str.get_dummies(',').add_prefix(col + '_')
        dummies['cust_id'] = trx['cust_id']
        dummies = dummies.groupby('cust_id').sum().reset_index()
        multi_features.append(dummies)
    # Merge multi-label features
    for f in multi_features:
        customer_features = customer_features.merge(f, on='cust_id', how='left')
    
    # Multinomial features
    multinomial_labels = ['prod_type_1','prod_insole']
    for col in multinomial_labels:
        dummies = (
            trx[col]
            .fillna('unknown')
            .astype(str)
            .str.strip()
            .str.get_dummies()
            .add_prefix(col + '_')
        )
        dummies['cust_id'] = trx['cust_id']
        dummies = dummies.groupby('cust_id').sum().reset_index()
        customer_features = customer_features.merge(dummies, on='cust_id', how='left')

    top_brands = trx["prod_brand"].value_counts().head(10).index.tolist() if "prod_brand" in trx.columns else []

    if len(top_brands) > 0:
        brand_counts = (
            trx[trx["prod_brand"].isin(top_brands)]
            .groupby(["cust_id", "prod_brand"])
            .size()
            .unstack(fill_value=0)
        )
        brand_counts.columns = [f"brand_{str(c)}" for c in brand_counts.columns]
        customer_features = customer_features.merge(brand_counts.reset_index(), on="cust_id", how="left")

    # Numerber of returns and average outlet
    returned = trx.groupby('cust_id')['returned_to_shop_id'].apply(
        lambda x: x.notna().sum()
    ).reset_index(name='returned_items')
    customer_features = customer_features.merge(returned, on='cust_id', how='left')
    outlet = trx.groupby('cust_id')['prod_outlet'].mean().reset_index()
    customer_features = customer_features.merge(outlet, on='cust_id', how='left')

    # Return rate and active-month ratio
    customer_features['return_rate'] = customer_features['returned_items'] / customer_features['n_lines'].replace(0, np.nan)
    tenure_months = (customer_features['tenure_days'] / 30.44).clip(lower=1)
    customer_features['active_month_ratio'] = customer_features['active_months'] / tenure_months

    # Encode heel height
    heel_map = {'<2.5 cm':0, '2.5-5 cm':1, '5-8 cm':2, '>8 cm':3}
    trx['prod_heel_encoded'] = trx['prod_heel'].map(heel_map).fillna(-1)
    heel = trx.groupby('cust_id')['prod_heel_encoded'].mean().reset_index()
    customer_features = customer_features.merge(heel, on='cust_id', how='left')
    print("Basic features created.")

    customer_features = add_window_features(trx, customer_features, reference_date=trx['order_date'].max())
    print("Window features created.")
    customer_features = add_cadence_features(trx, customer_features)
    print("Cadence features created.")
    customer_features = add_diversity_features(trx, customer_features)
    print("Diversity features created.")

    # Merge with original customer list to keep all customers in the dataset
    df = data[['cust_id']].merge(customer_features, on='cust_id', how='left')

    # customers without history
    df = df.fillna(0)
    return df



In [ ]:
# Build features once for all customers seen in transactions
all_customers = transactions[["cust_id"]].drop_duplicates().reset_index(drop=True)
all_features = build_features(all_customers, transactions)

# Split by customer lists from CLV files
train_features = train_target[["cust_id"]].merge(all_features, on="cust_id", how="inner")
test_features = test_target[["cust_id"]].merge(all_features, on="cust_id", how="left").fillna(0)

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_features.to_csv(ALL_OUT_PATH, index=False)
train_features.to_csv(TRAIN_OUT_PATH, index=False)
test_features.to_csv(TEST_OUT_PATH, index=False)

print(f"Saved: {TRAIN_OUT_PATH} -> {train_features.shape}")
print(f"Saved: {TEST_OUT_PATH} -> {test_features.shape}")
print(f"Saved: {ALL_OUT_PATH} -> {all_features.shape}")


drop_duplicates   : 1,063 removed (343,149 remain)


In [8]:
# Audit: multi-label and multinomial feature quality (STRICT KEEP / REVIEW / DROP)
# Run this after loading transactions and train_target.

multi_labels = [
    'prod_type_3', 'prod_type_4', 'prod_type_5',
    'prod_material', 'prod_print',
    'prod_comfort_sole', 'prod_comfort_wear', 'prod_clasp'
]
multinomial_labels = ['prod_type_1', 'prod_insole']

def _group_quality_strict(trx, train_target, col, is_multilabel=True):
    base = train_target[['cust_id', 'revenue_2018_2019']].copy()

    if is_multilabel:
        tokens = (
            trx[col].dropna().astype(str)
            .str.split(',')
            .explode()
            .astype(str)
            .str.strip()
        )
        cleaned = (
            trx[col].fillna('unknown').astype(str)
            .str.split(',').apply(lambda vals: ','.join(v.strip() for v in vals))
        )
        dummies = cleaned.str.get_dummies(',').add_prefix(col + '_')
    else:
        tokens = trx[col].dropna().astype(str).str.strip()
        dummies = (
            trx[col].fillna('unknown').astype(str).str.strip()
            .str.get_dummies().add_prefix(col + '_')
        )

    tokens = tokens[tokens != '']
    non_null_rate = float(trx[col].notna().mean())
    n_unique_raw = int(tokens.nunique())

    dummies['cust_id'] = trx['cust_id']
    cust = dummies.groupby('cust_id').sum().reset_index()
    df = base.merge(cust, on='cust_id', how='left').fillna(0)

    feat_cols = [c for c in df.columns if c.startswith(col + '_') and df[c].nunique() > 1]
    if not feat_cols:
        return {
            'column': col,
            'type': 'multilabel' if is_multilabel else 'multinomial',
            'non_null_rate': non_null_rate,
            'n_unique_raw': n_unique_raw,
            'n_features': 0,
            'median_nonzero_rate': 0.0,
            'max_abs_corr_all': 0.0,
            'max_abs_corr_no_unknown': 0.0,
            'unknown_nz_rate': 0.0,
            'recommendation': 'DROP',
            'reason': 'No non-constant derived features.'
        }

    nz = (df[feat_cols] != 0).mean()
    median_nz = float(nz.median())

    unknown_col = col + '_unknown'
    unknown_nz_rate = float(nz[unknown_col]) if unknown_col in nz.index else 0.0

    y = df['revenue_2018_2019'].to_numpy()
    corr_pairs = []
    for c in feat_cols:
        x = df[c].to_numpy()
        corr = np.corrcoef(x, y)[0, 1]
        if np.isnan(corr):
            corr = 0.0
        corr_pairs.append((c, abs(float(corr))))

    max_abs_corr_all = max(v for _, v in corr_pairs)
    non_unknown_pairs = [(c, v) for c, v in corr_pairs if not c.endswith('_unknown')]
    max_abs_corr_no_unknown = max((v for _, v in non_unknown_pairs), default=0.0)

    # Strict practical rules:
    # 1) Very sparse + weak signal (excluding unknown) -> DROP
    if median_nz < 0.01 and max_abs_corr_no_unknown < 0.12:
        rec, reason = 'DROP', 'Very sparse block with weak non-unknown signal.'
    # 2) Very low coverage + weak non-unknown signal -> DROP
    elif non_null_rate < 0.10 and max_abs_corr_no_unknown < 0.15:
        rec, reason = 'DROP', 'Low coverage and weak non-unknown signal.'
    # 3) High-cardinality sparse block with only moderate signal -> REVIEW
    elif len(feat_cols) >= 30 and median_nz < 0.02 and max_abs_corr_no_unknown < 0.20:
        rec, reason = 'REVIEW', 'High-cardinality and sparse; prune rare categories.'
    # 4) Signal dominated by unknown category -> REVIEW
    elif unknown_nz_rate > 0.50 and max_abs_corr_no_unknown < 0.18:
        rec, reason = 'REVIEW', 'Signal appears dominated by missingness (unknown bucket).'
    else:
        rec, reason = 'KEEP', 'Strict checks passed.'

    return {
        'column': col,
        'type': 'multilabel' if is_multilabel else 'multinomial',
        'non_null_rate': non_null_rate,
        'n_unique_raw': n_unique_raw,
        'n_features': len(feat_cols),
        'median_nonzero_rate': median_nz,
        'max_abs_corr_all': max_abs_corr_all,
        'max_abs_corr_no_unknown': max_abs_corr_no_unknown,
        'unknown_nz_rate': unknown_nz_rate,
        'recommendation': rec,
        'reason': reason,
    }

report = []
for c in multi_labels:
    report.append(_group_quality_strict(transactions, train_target, c, is_multilabel=True))
for c in multinomial_labels:
    report.append(_group_quality_strict(transactions, train_target, c, is_multilabel=False))

order_map = {'DROP': 0, 'REVIEW': 1, 'KEEP': 2}
report_df = pd.DataFrame(report)
report_df['_ord'] = report_df['recommendation'].map(order_map)
report_df = report_df.sort_values(['_ord', 'max_abs_corr_no_unknown'], ascending=[True, False]).drop(columns=['_ord'])

print(report_df.to_string(index=False))

print('')
print('Quick read (STRICT):')
for _, r in report_df.iterrows():
    print(
        f"- {r['column']}: {r['recommendation']} | "
        f"non_null={r['non_null_rate']:.3f}, features={int(r['n_features'])}, "
        f"median_nz={r['median_nonzero_rate']:.3f}, "
        f"corr_no_unknown={r['max_abs_corr_no_unknown']:.3f}"
    )




           column        type  non_null_rate  n_unique_raw  n_features  median_nonzero_rate  max_abs_corr_all  max_abs_corr_no_unknown  unknown_nz_rate recommendation                                                    reason
      prod_type_4  multilabel       0.413870            30          31             0.009160          0.368920                 0.176630         0.679778         REVIEW       High-cardinality and sparse; prune rare categories.
      prod_type_5  multilabel       0.981814            51          52             0.019942          0.166716                 0.166716         0.036075         REVIEW       High-cardinality and sparse; prune rare categories.
prod_comfort_wear  multilabel       0.094480             4           5             0.047319          0.395291                 0.157864         0.942508         REVIEW Signal appears dominated by missingness (unknown bucket).
prod_comfort_sole  multilabel       0.122422             7           8             0.028844         